In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Birthdate").getOrCreate()
df_people=spark.read.parquet("/databricks-datasets/learning-spark-v2/people/people-10m.parquet/")
df_people.printSchema()

In [0]:
from pyspark.sql.functions import window,desc,col
df_people.orderBy(col("birthDate")).show()


In [0]:
df_people.groupby("birthDate").count().orderBy(desc("birthDate")).show()

In [0]:
df_people_count_datewise=df_people.groupby("birthDate").count().orderBy(desc("birthDate"))
df_people_count_datewise.show()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number,lit,count,round,timestamp_add
window_spec=Window.partitionBy("birthDate").orderBy("id")
window_spec_all=Window.partitionBy(col("birthDate"))
df_people.withColumn("rown",row_number().over(window_spec)).withColumn("total_cnt",count(col("birthDate")).over(window_spec_all)).show()

In [0]:
df_people.withColumn("rown",row_number().over(window_spec)).withColumn("total_cnt",count(col("birthDate")).over(window_spec_all)).printSchema()

In [0]:
df_people.withColumn("rown",row_number().over(window_spec)).\
            withColumn("total_cnt",count(col("birthDate")).over(window_spec_all)).\
                withColumn("added_seconds",round((86400/col("total_cnt"))*col("rown"))).\
                    withColumn("new_date",timestamp_add("SECOND",col("added_seconds"),col("birthDate"))).show()

In [0]:
df_people=df_people.withColumn("rown",row_number().over(window_spec)).\
            withColumn("total_cnt",count(col("birthDate")).over(window_spec_all)).\
                withColumn("added_seconds",round((86400/col("total_cnt"))*col("rown"))).\
                    withColumn("new_date",timestamp_add("SECOND",col("added_seconds"),col("birthDate"))).\
                        select("id","firstName","lastName","birthDate","new_date")

In [0]:
df_people.show()

In [0]:
from pyspark.sql.functions import to_date

In [0]:
df_people.where((col("birthDate")> to_date(lit("1952-01-02"))) & (col("birthDate") < to_date(lit("1952-01-03")))).show()

In [0]:
display(df_people.where((col("birthDate")> to_date(lit("1952-01-02"))) & (col("birthDate") < to_date(lit("1952-01-03")))))

In [0]:
df_people.where((col("birthDate")> to_date(lit("1952-01-03"))) & (col("birthDate") < to_date(lit("1952-01-04")))).count()

In [0]:
df_people_count_datewise=df_people.groupby("birthDate").count().orderBy("birthDate").show()

In [0]:
# check any lag
from pyspark.sql.window import Window
window_spec_only_orderby=Window.orderBy("birthDate")
window_spec_only_orderbyPartitionby=Window.partitionBy("birthDate").orderBy("id")
window_spec_only_orderby

In [0]:
from pyspark.sql.functions import lag

In [0]:
df_people.withColumn("Date_lag",lag("birthdate",1).over(window_spec_only_orderby)).withColumn("Date_lag_with_partion",lag("birthDate",1).over(window_spec_only_orderbyPartitionby)).show()

In [0]:
df_people.withColumn("Date_lag",lag("birthdate",1).over(window_spec_only_orderby)).\
         withColumn("Date_lag_with_partion",lag("birthDate",1).over(window_spec_only_orderbyPartitionby)).\
         where(to_date("birthdate") > '1952-01-01').show()

In [0]:
from pyspark.sql.functions import lag,dense_rank

In [0]:
dr_window_spec= Window.orderBy("birthDate")
df_people.withColumn("DR_BDATE",dense_rank().over(dr_window_spec)).show()

In [0]:
df_people_date_distinct=df_people.withColumn("DR_BDATE",dense_rank().over(dr_window_spec)).select("birthDate","DR_BDATE").distinct()

In [0]:
df_people_date_distinct.show()

In [0]:
df_people_date_distinct.withColumn("prev_date",lag("birthDate",1).over(dr_window_spec)).show()

In [0]:
from pyspark.sql.functions import date_diff


In [0]:
df_people_date_distinct.withColumn("prev_date",lag("birthDate",1).over(dr_window_spec)).withColumn("date_difference",date_diff("birthDate","prev_date")).where(col("date_difference") == 1).show()

In [0]:
df_people_date_distinct=df_people_date_distinct.withColumn("prev_date",lag("birthDate",1).over(dr_window_spec))

In [0]:
df_people.join(df_people_date_distinct,on="birthDate",how="left").orderBy("DR_BDATE").where(to_date("birthDate") =='1952-01-02').show()